In [1]:
import pandas as pd
import numpy as np
import json

## Define the metadata

In [2]:
organism = "homo_sapiens"
cell_source = "lung"
cell_state = None

reference = "GRCh38"
paper_doi = "https://doi.org/10.1126/sciadv.aba1983"
table_link = "https://www.science.org/doi/suppl/10.1126/sciadv.aba1983/suppl_file/aba1983_data_s2.txt"

# don't include in header
table_names = ["downloadSupplement.txt", "downloadSupplement2.txt", "downloadSupplement3.txt", "downloadSupplement4.txt", "downloadSupplement5.txt"] 

## Read in file

In [12]:
#excel = pd.read_excel(table_name, sheet_name = ["Degs human_limb&bone Fig. 1d", "Degs human long bone Fig. S3b", "Degs human calvaria Fig. 6b"], skiprows = 0)
tables = []
for tn in table_names:
    tables.append(pd.read_csv(tn, sep = "\t"))
df = pd.concat(tables)
df.rename(columns = {"wilcox.p": "p_val_adj"}, inplace=True)

df

,cellType,gene,log1p_FC,pct.inGroup,pct.outGroup,p_val_adj,wilcox.fdr,wilcox.bonferroni
1,Basal,MIR205HG,1.757514,0.969231,0.489669,8.086494e-39,2.838360e-36,2.838360e-36
2,Basal,KRT5,0.750194,0.861538,0.264463,3.598166e-36,6.314782e-34,1.262956e-33
3,Basal,EYA2,1.436151,0.984615,0.669421,1.463281e-33,1.712039e-31,5.136117e-31
4,Basal,CYP24A1,0.678724,0.892308,0.320248,4.873706e-32,4.276677e-30,1.710671e-29
5,Basal,KRT17,1.943518,0.953846,0.526860,2.228007e-31,1.564061e-29,7.820306e-29
...,...,...,...,...,...,...,...,...
2924,ILC B,PARP8,-0.325149,0.923077,0.944015,1.614698e-03,1.637124e-03,4.714917e-01
2925,ILC B,PHF20,0.315690,0.969231,0.978764,1.810093e-03,1.828883e-03,5.285472e-01
2926,ILC B,TOB1,0.321810,0.815385,0.870656,8.323321e-03,8.323321e-03,1.000000e+00
2927,ILC B,PRKG1,0.351816,0.553846,0.669884,4.134930e-03,4.163446e-03,1.000000e+00


In [13]:
df.head()

,cellType,gene,log1p_FC,pct.inGroup,pct.outGroup,p_val_adj,wilcox.fdr,wilcox.bonferroni
1,Basal,MIR205HG,1.757514,0.969231,0.489669,8.086494e-39,2.838360e-36,2.838360e-36
2,Basal,KRT5,0.750194,0.861538,0.264463,3.598166e-36,6.314782e-34,1.262956e-33
3,Basal,EYA2,1.436151,0.984615,0.669421,1.463281e-33,1.712039e-31,5.136117e-31
4,Basal,CYP24A1,0.678724,0.892308,0.320248,4.873706e-32,4.276677e-30,1.710671e-29
5,Basal,KRT17,1.943518,0.953846,0.526860,2.228007e-31,1.564061e-29,7.820306e-29


## Perform the filtering

In [14]:
col_pval = "p_val_adj"
col_lfc = "log1p_FC"
col_gene = "gene"
col_ct = "cellType" 
col_rank = "pct.inGroup" # set to None if pct. 1DNE

In [15]:
min_mean = 100
max_pval = 1e-10
min_lfc = 2.2
max_gene_shares = 2
max_per_celltype = 20

# filter by pval and lfc
dfc = df.query(f"{col_pval} <= {max_pval} & {col_lfc} >= {min_lfc}")

# mask out genes that are shared between max_gene_shares cell types
non_repeat_genes = dfc[col_gene].value_counts()[dfc[col_gene].value_counts() < max_gene_shares].index.values

m = dfc[dfc[col_gene].isin(non_repeat_genes)]

if col_rank:
    m = m.sort_values(col_rank, ascending = True)

# max number to sample is equal to the min number of genes across all celltype
n_sample = min(m[col_ct].value_counts().max(), max_per_celltype)

# sample n_sample genes
markers = m.groupby(col_ct).tail(n_sample)
markers_dict = markers.groupby(col_ct)[col_gene].apply(lambda x: list(x)).to_dict()

## Check how many genes per cell type

In [16]:
markers

,cellType,gene,log1p_FC,pct.inGroup,pct.outGroup,p_val_adj,wilcox.fdr,wilcox.bonferroni
1022,AT1,NTM,2.208726,0.833333,0.687631,3.074875e-18,1.079376e-17,2.093990e-15
5881,PNEC,SEC11C,2.493893,0.875000,0.746667,9.400886e-11,1.405860e-09,6.185783e-08
5838,PNEC,GRP,3.643086,0.875000,0.009524,3.142829e-86,2.067981e-83,2.067981e-83
1165,B Plasma,IGHA1,2.316354,0.911765,0.559223,8.953227e-32,1.934440e-30,6.383651e-29
849,AT1,NCKAP5,2.210212,0.944444,0.710692,4.801476e-35,1.557050e-33,3.269805e-32
358,AT2,SFTPC,3.597459,0.945946,0.602105,6.564154e-37,4.473002e-35,3.131101e-34
3479,Mast,IL1RL1,2.302870,0.954545,0.384117,2.593922e-43,2.564741e-41,2.051793e-40
1136,B Plasma,JCHAIN,3.162226,0.955882,0.413592,5.750390e-42,1.025007e-39,4.100028e-39
5853,PNEC,NRG1,2.798002,0.958333,0.809524,2.566611e-14,1.055519e-12,1.688830e-11
2833,DC Langerhans,S100B,2.757214,0.960000,0.630332,7.566173e-29,9.730098e-27,4.865049e-26


## Find Ensembl ID


In [17]:
import sys
import time

from urllib.parse import urlparse, urlencode
from urllib.request import urlopen, Request
from urllib.error import HTTPError

In [18]:
class EnsemblRestClient(object):
    def __init__(self, server='http://rest.ensembl.org', reqs_per_sec=5):
        self.server = server
        self.reqs_per_sec = reqs_per_sec
        self.req_count = 0
        self.last_req = 0

    def perform_rest_action(self, endpoint, hdrs=None, params=None):
        if hdrs is None:
            hdrs = {}

        if 'Content-Type' not in hdrs:
            hdrs['Content-Type'] = 'application/json'

        if params:
            endpoint += '?' + urlencode(params)

        data = None

        # check if we need to rate limit ourselves
        if self.req_count >= self.reqs_per_sec:
            delta = time.time() - self.last_req
            if delta < 1:
                time.sleep(1 - delta)
            self.last_req = time.time()
            self.req_count = 0
        
        try:
            request = Request(self.server + endpoint, headers=hdrs)
            response = urlopen(request)
            content = response.read()
            if content:
                data = json.loads(content)
            self.req_count += 1

        except HTTPError as e:
            # check if we are being rate limited by the server
            if e.code == 429:
                if 'Retry-After' in e.headers:
                    retry = e.headers['Retry-After']
                    time.sleep(float(retry))
                    self.perform_rest_action(endpoint, hdrs, params)
            else:
                sys.stderr.write('Request failed for {0}: Status code: {1.code} Reason: {1.reason}\n'.format(endpoint, e))
           
        return data

    def get_id(self, species, symbol):
        genes = self.perform_rest_action(
            endpoint='/xrefs/symbol/{0}/{1}'.format(species, symbol), 
            params={'object_type': 'gene'}
        )
        if genes:
            stable_id = genes[0]['id']
            return stable_id
        else:
            return "gene not found"

In [19]:
def run(symbol):
    client = EnsemblRestClient()
    gene_id = client.get_id('human', symbol)
    return gene_id

## Convert to evidence json


In [20]:
df = markers[[col_ct, col_gene]].rename(columns={col_ct : "cell_type_label", col_gene: "gene"})
df["organism"] = organism
df["cell_source"] = cell_source
df["cell_state"] = cell_state
df["gene_id"] = df["gene"]
df["gene_id"] = df["gene_id"].apply(run)
save = df.to_dict(orient = "records")

In [21]:
save

[{'cell_type_label': 'AT1',
  'gene': 'NTM',
  'organism': 'homo_sapiens',
  'cell_source': 'lung',
  'cell_state': None,
  'gene_id': 'ENSG00000182667'},
 {'cell_type_label': 'PNEC',
  'gene': 'SEC11C',
  'organism': 'homo_sapiens',
  'cell_source': 'lung',
  'cell_state': None,
  'gene_id': 'ENSG00000166562'},
 {'cell_type_label': 'PNEC',
  'gene': 'GRP',
  'organism': 'homo_sapiens',
  'cell_source': 'lung',
  'cell_state': None,
  'gene_id': 'ENSG00000165623'},
 {'cell_type_label': 'B Plasma',
  'gene': 'IGHA1',
  'organism': 'homo_sapiens',
  'cell_source': 'lung',
  'cell_state': None,
  'gene_id': 'ENSG00000211895'},
 {'cell_type_label': 'AT1',
  'gene': 'NCKAP5',
  'organism': 'homo_sapiens',
  'cell_source': 'lung',
  'cell_state': None,
  'gene_id': 'ENSG00000176771'},
 {'cell_type_label': 'AT2',
  'gene': 'SFTPC',
  'organism': 'homo_sapiens',
  'cell_source': 'lung',
  'cell_state': None,
  'gene_id': 'ENSG00000168484'},
 {'cell_type_label': 'Mast',
  'gene': 'IL1RL1',
  'o

## Save evidence

In [22]:
with open("evidence.json", "w") as f:
    json.dump(save, f, indent = 4) 